In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, BooleanType
from datetime import date, timedelta

SILVER_TABLE = "nyc_taxi.silver.trips_clean"
ZONE_LOOKUP_PATH = "/Volumes/nyc_taxi/bronze/nyc_taxi_raw/zone/taxi_zone_lookup.csv"

DIM_DATE_TABLE = "nyc_taxi.gold.dim_date"
DAILY_SUMMARY_TABLE = "nyc_taxi.gold.daily_summary"
HOURLY_DEMAND_TABLE = "nyc_taxi.gold.hourly_demand"
ZONE_PERFORMANCE_TABLE = "nyc_taxi.gold.zone_performance"

In [0]:
df = spark.read.table(SILVER_TABLE)
print(f"Silver rows: {df.count()}")

Silver rows: 11634221


In [0]:
bounds = df.select(
    F.min(F.to_date("tpep_pickup_datetime")).alias("min_date"),
    F.max(F.to_date("tpep_pickup_datetime")).alias("max_date")
).first()

start = bounds["min_date"]
end = bounds["max_date"]
num_days = (end - start).days + 1

dates = [(start + timedelta(days = i),) for i in range(num_days)]
df_dates = spark.createDataFrame(dates, ["date"])

df_dim_date = (
    df_dates
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
    .withColumn("quarter", F.quarter("date"))
    .select("date_key", "date", "year", "month", "month_name",
            "day_of_month", "day_of_week", "day_name", "is_weekend", "quarter")
)

print(f"dim_date_row {df_dim_date.count()}")

dim_date_row 91


In [0]:
df_daily = (
    df
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes"),
        F.countDistinct("PULocationID").alias("unique_pickup_zones")
    )
    .withColumn("date_key", F.date_format("pickup_date", "yyyyMMdd").cast("int"))
    .select(
        "pickup_date", "date_key", "total_trips", "total_revenue",
        "avg_fare", "avg_tip", "avg_trip_distance",
        "avg_trip_duration_minutes", "unique_pickup_zones"
    )
    .orderBy("pickup_date")
)
print(f"daily_summary_rows {df_daily.count()}")

daily_summary_rows 91


In [0]:
df_hourly = (
    df
    .groupBy("pickup_day_of_week", "pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("tip_percentage"), 2).alias("avg_tip_percentage")
    )
    .withColumn(
        "day_name",
        F.when(F.col("pickup_day_of_week") == 1, "Sunday")
        .when(F.col("pickup_day_of_week") == 2, "Monday")
        .when(F.col("pickup_day_of_week") == 3, "Tuesday")
        .when(F.col("pickup_day_of_week") == 4, "Wednesday")
        .when(F.col("pickup_day_of_week") == 5, "Thursday")
        .when(F.col("pickup_day_of_week") == 6, "Friday")
        .otherwise("Saturday")
    )
    .select("pickup_day_of_week", "day_name", "pickup_hour", "total_trips", "avg_trip_distance",
            "avg_trip_duration_minutes", "avg_fare_amount", "avg_tip_percentage")
    .orderBy("pickup_day_of_week", "pickup_hour")
)
print(f"hourly_demand_rows {df_hourly.count()}")

hourly_demand_rows 168


In [0]:
df_zones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ZONE_LOOKUP_PATH)
    .select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("zone_name"),
        F.col("Borough").alias("borough"),
        F.col("service_zone")
    )
)

df_zone_agg = (
    df
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
)

df_zone_pf = (
    df_zone_agg
    .join(df_zones, on = "PULocationID", how = "left")
    .withColumn("zone_name", F.coalesce(F.col("zone_name"), F.lit("Unknown")))
    .withColumn("borough", F.coalesce(F.col("borough"), F.lit("Unknown")))
    .withColumn("service_zone", F.coalesce(F.col("service_zone"), F.lit("Unknown")))
    .select(
        "PULocationID", "zone_name", "borough", "service_zone",
        "total_trips", "total_revenue", "avg_fare", "avg_tip",
        "avg_trip_distance", "avg_trip_duration_minutes"
    )
    .orderBy(F.desc("total_trips"))
)
print(f"zone_performance_rows: {df_zone_pf.count()}")

zone_performance_rows: 261


In [0]:
df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_DATE_TABLE)
df_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DAILY_SUMMARY_TABLE)
df_hourly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(HOURLY_DEMAND_TABLE)
df_zone_pf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(ZONE_PERFORMANCE_TABLE)

print("All 4 Gold tables created")

All 4 Gold tables created


In [0]:
expected = {
    DIM_DATE_TABLE: 91,
    DAILY_SUMMARY_TABLE: 91,
    HOURLY_DEMAND_TABLE: 168,
}

for table, exp_count in expected.items():
    actual = spark.read.table(table).count()
    assert actual == exp_count, f"{table}: expected {exp_count}, got {actual}"
    print(f"✅ {table}: {actual} rows")

# zone_performance has no fixed expected count
zp_count = spark.read.table(ZONE_PERFORMANCE_TABLE).count()
print(f"✅ {ZONE_PERFORMANCE_TABLE}: {zp_count} rows (variable)")

✅ nyc_taxi.gold.dim_date: 91 rows
✅ nyc_taxi.gold.daily_summary: 91 rows
✅ nyc_taxi.gold.hourly_demand: 168 rows
✅ nyc_taxi.gold.zone_performance: 261 rows (variable)
